# Matrix Factorization for Recommender Systems

## Learning Objectives

1. **Formulate** rating prediction as matrix factorization
2. **Implement** SGD training with bias terms
3. **Evaluate** on held-out ratings
4. **Contrast** with neighborhood-based methods

## The Latent Factor Model

Instead of explicit neighborhoods, learn **latent representations**:

$$\hat{r}_{ui} = \mu + b_u + c_i + \mathbf{p}_u \cdot \mathbf{q}_i$$

- $\mu$: global mean rating
- $b_u$: user bias (e.g., harsh rater → negative)
- $c_i$: item bias (e.g., popular item → positive)
- $\mathbf{p}_u \in \mathbb{R}^k$: user latent factors
- $\mathbf{q}_i \in \mathbb{R}^k$: item latent factors

**Regularized objective:**
$$\min_{\mu,b,c,P,Q} \sum_{(u,i)\in\Omega} (r_{ui} - \hat{r}_{ui})^2 + \lambda(\|\mathbf{p}_u\|^2 + \|\mathbf{q}_i\|^2 + b_u^2 + c_i^2)$$

In [1]:
import math, random
from collections import defaultdict

def dot(a, b): return sum(x*y for x,y in zip(a,b))

class MatrixFactorization:
    def __init__(self, k=10, eta=0.01, lam=0.02, n_epochs=50, seed=0):
        self.k = k; self.eta = eta; self.lam = lam; self.n_epochs = n_epochs
        self.rng = random.Random(seed)

    def fit(self, train_ratings):
        """train_ratings: list of (user, item, rating)"""
        users = list(set(u for u,_,_ in train_ratings))
        items = list(set(i for _,i,_ in train_ratings))
        self.users = {u:idx for idx,u in enumerate(users)}
        self.items = {i:idx for idx,i in enumerate(items)}
        m, n = len(users), len(items)
        self.mu = sum(r for _,_,r in train_ratings) / len(train_ratings)
        self.b_u = [0.0]*m; self.c_i = [0.0]*n
        self.P = [[self.rng.gauss(0, 0.1) for _ in range(self.k)] for _ in range(m)]
        self.Q = [[self.rng.gauss(0, 0.1) for _ in range(self.k)] for _ in range(n)]
        self.losses = []
        obs = list(train_ratings)
        for epoch in range(self.n_epochs):
            self.rng.shuffle(obs)
            total_sq = 0.0
            for u, i, r in obs:
                uid, iid = self.users[u], self.items[i]
                pred = self.mu + self.b_u[uid] + self.c_i[iid] + dot(self.P[uid], self.Q[iid])
                e = r - pred
                total_sq += e**2
                # Update biases
                self.b_u[uid] += self.eta * (e - self.lam * self.b_u[uid])
                self.c_i[iid] += self.eta * (e - self.lam * self.c_i[iid])
                # Update factors
                for l in range(self.k):
                    pu, qi = self.P[uid][l], self.Q[iid][l]
                    self.P[uid][l] += self.eta * (e*qi - self.lam*pu)
                    self.Q[iid][l] += self.eta * (e*pu - self.lam*qi)
            self.losses.append(math.sqrt(total_sq / len(obs)))

    def predict(self, u, i):
        if u not in self.users or i not in self.items: return self.mu
        uid, iid = self.users[u], self.items[i]
        return self.mu + self.b_u[uid] + self.c_i[iid] + dot(self.P[uid], self.Q[iid])

# Synthetic rating data
rng = random.Random(42)
users_list = [f'U{i}' for i in range(50)]
items_list = [f'M{j}' for j in range(30)]
# True latent factors
P_true = {u: [rng.gauss(0,1) for _ in range(3)] for u in users_list}
Q_true = {i: [rng.gauss(0,1) for _ in range(3)] for i in items_list}
all_ratings = [(u, i, max(1, min(5, round(dot(P_true[u], Q_true[i]) + rng.gauss(0,0.5)))))
               for u in users_list for i in items_list if rng.random() < 0.4]

# Train/test split
rng.shuffle(all_ratings)
split = int(len(all_ratings)*0.8)
train, test = all_ratings[:split], all_ratings[split:]

mf = MatrixFactorization(k=5, eta=0.01, lam=0.02, n_epochs=100)
mf.fit(train)
print(f"Training RMSE progression:")
for ep in [0, 9, 24, 49, 99]:
    print(f"  Epoch {ep+1:>3}: {mf.losses[ep]:.4f}")

Training RMSE progression:
  Epoch   1: 0.6511
  Epoch  10: 0.5698
  Epoch  25: 0.5394
  Epoch  50: 0.4502
  Epoch 100: 0.2602


In [2]:
# Evaluate on test set
test_preds = [(r, mf.predict(u,i)) for u,i,r in test]
test_rmse = math.sqrt(sum((r-p)**2 for r,p in test_preds)/len(test_preds))
print(f"""
Test RMSE: {test_rmse:.4f}""")

# Baseline: always predict global mean
baseline_rmse = math.sqrt(sum((r-mf.mu)**2 for u,i,r in test)/len(test))
print(f"Baseline RMSE (global mean): {baseline_rmse:.4f}")
print(f"Improvement: {(baseline_rmse-test_rmse)/baseline_rmse*100:.1f}%")

# Show sample predictions
print("""
Sample predictions vs actual:""")
print(f"""{'User':>4} {'Item':>4} {'Actual':>8} {'Predicted':>10}""")
for u,i,r in test[:8]:
    pred = mf.predict(u,i)
    print(f"{u:>4} {i:>4} {r:>8} {pred:>10.3f}")


Test RMSE: 0.8423
Baseline RMSE (global mean): 0.7057
Improvement: -19.4%

Sample predictions vs actual:
User Item   Actual  Predicted
 U12  M27        1      1.062
 U46  M24        1      1.480
 U30  M25        1      1.273
 U29  M10        1      0.713
 U35  M23        1      1.668
  U7  M11        1      1.193
 U27   M4        1      0.992
 U46  M21        1      2.228


## Comparison: Neighborhood vs Matrix Factorization

| Method | Training | Prediction | Cold start | Interpretability |
|--------|----------|-----------|-----------|-----------------|
| User-user CF | $O(m^2 n)$ | $O(m)$ per item | Poor | High |
| Item-item CF | $O(n^2 m)$ | $O(k)$ | Poor for new items | Medium |
| Matrix factorization | $O(|\Omega| k)$ | $O(k)$ | Poor | Low |

**Why matrix factorization won the Netflix Prize:**
- Handles sparsity better than neighborhood methods
- Regularization prevents overfitting
- Easily extended with implicit feedback, temporal dynamics, and biases
- Scales with SGD to millions of users/items